# Treino rigoroso de talhões (Colab)

Fluxo sem clone: grava scripts locais e executa treino + prova real.

In [ ]:
# 1) Dependências
!pip -q install tensorflow matplotlib

In [ ]:
# 2) Montar Drive (obrigatório antes de rodar scripts .py)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Gravar script de treino
%%writefile treinamento_seguro_unet.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base

    raise RuntimeError(
        "Drive não montado. No Colab, execute antes: \n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


pasta_base = garantir_drive_montado()

KERNEL_SIZE = 128
READ_SIZE = 129
BATCH_SIZE = 32
EPOCHS = 80
SEED = 42
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'


def localizar_tfrecord(base_dir: str) -> str:
    busca = glob.glob(os.path.join(base_dir, '*MASSIVE*tfrecord*'))
    if not busca:
        busca = glob.glob(os.path.join(base_dir, '*tfrecord*'))
        busca.sort(key=os.path.getmtime, reverse=True)
    if not busca:
        raise FileNotFoundError(f'Nenhum TFRecord encontrado em {base_dir}')
    return busca[0]


def normalizar_banda(img: tf.Tensor) -> tf.Tensor:
    min_v = tf.reduce_min(img)
    max_v = tf.reduce_max(img)
    return tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))


def parse_and_process(example_proto):
    features_dict = {
        band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]
    }
    parsed = tf.io.parse_single_example(example_proto, features_dict)

    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        img = normalizar_banda(img)
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)

    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image_stacked, lbl


def augment(image, label):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        label = tf.image.flip_left_right(label)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        label = tf.image.flip_up_down(label)
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    label = tf.image.rot90(label, k)
    return image, label


def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice_loss = 1.0 - dice_coef(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss


def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


def build_unet(input_shape):
    inputs = layers.Input(shape=input_shape)

    c1 = conv_block(inputs, 32)
    p1 = layers.MaxPooling2D()(c1)

    c2 = conv_block(p1, 64)
    p2 = layers.MaxPooling2D()(c2)

    c3 = conv_block(p2, 128)
    p3 = layers.MaxPooling2D()(c3)

    c4 = conv_block(p3, 256)

    u5 = layers.Conv2DTranspose(128, 2, strides=2, padding='same')(c4)
    u5 = layers.concatenate([u5, c3])
    c5 = conv_block(u5, 128)

    u6 = layers.Conv2DTranspose(64, 2, strides=2, padding='same')(c5)
    u6 = layers.concatenate([u6, c2])
    c6 = conv_block(u6, 64)

    u7 = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(c6)
    u7 = layers.concatenate([u7, c1])
    c7 = conv_block(u7, 32)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c7)
    return models.Model(inputs=[inputs], outputs=[outputs])


caminho_arquivo = localizar_tfrecord(pasta_base)
print(f"📂 Lendo dados de: {caminho_arquivo}")

print("🔢 Verificando tamanho do arquivo...")
raw_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP')
N_REAL = sum(1 for _ in raw_dataset)
print(f"✅ Total de amostras: {N_REAL}")

N_TRAIN = int(N_REAL * 0.8)
full_dataset = tf.data.TFRecordDataset(caminho_arquivo, compression_type='GZIP').map(
    parse_and_process, num_parallel_calls=tf.data.AUTOTUNE
)
full_dataset = full_dataset.shuffle(max(N_REAL, 1), seed=SEED, reshuffle_each_iteration=False)

train_ds = full_dataset.take(N_TRAIN).map(augment, num_parallel_calls=tf.data.AUTOTUNE).cache().shuffle(max(N_TRAIN, 1), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = full_dataset.skip(N_TRAIN).cache().batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

model = build_unet((KERNEL_SIZE, KERNEL_SIZE, len(INPUT_BANDS)))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=bce_dice_loss,
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name='iou'),
        dice_coef,
    ],
)

checkpoint_last = os.path.join(pasta_base, 'Modelo_Checkpoint_last.keras')
checkpoint_best = os.path.join(pasta_base, 'Modelo_Checkpoint_best.keras')
csv_log = os.path.join(pasta_base, 'historico_treinamento.csv')
final_path = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')

cbs = [
    callbacks.ModelCheckpoint(checkpoint_last, save_best_only=False, verbose=1),
    callbacks.ModelCheckpoint(checkpoint_best, monitor='val_iou', mode='max', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_iou', mode='max', patience=12, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.CSVLogger(csv_log),
]

print("🔥 Iniciando Retreinamento Rigoroso (foco em talhões)...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cbs,
)

model.save(final_path)
print(f"✅ SUCESSO! Modelo final salvo em: {final_path}")
print(f"✅ Melhor checkpoint salvo em: {checkpoint_best}")


In [ ]:
# 4) Gravar script de validação
%%writefile validacao_visual_modelo.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import tensorflow as tf
import matplotlib.pyplot as plt


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base

    raise RuntimeError(
        "Drive não montado. No Colab, execute antes: \n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


print('--- INICIANDO PROVA REAL ---')
pasta_base = garantir_drive_montado()
caminho_modelo = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')

if not os.path.exists(caminho_modelo):
    raise FileNotFoundError(f'Modelo não encontrado: {caminho_modelo}')

model = tf.keras.models.load_model(caminho_modelo, compile=False)
print(f'✅ Modelo carregado: {caminho_modelo}')

busca_dados = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca_dados:
    raise FileNotFoundError('Nenhum TFRecord MASSIVE encontrado para validação visual.')

caminho_dados = busca_dados[0]
KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'


def parse_fast(example_proto):
    features_dict = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features_dict)
    inputs_list = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        min_v = tf.reduce_min(img)
        max_v = tf.reduce_max(img)
        img = tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))
        inputs_list.append(img)

    image_stacked = tf.concat(inputs_list, axis=-1)
    dense_lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(dense_lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image_stacked, lbl


dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP').map(parse_fast).batch(10).take(1)
imgs, labels = next(iter(dataset))
preds = model.predict(imgs, verbose=0)
preds_bin = (preds > 0.5).astype('float32')

intersection = (preds_bin * labels.numpy()).sum(axis=(1, 2, 3))
union = ((preds_bin + labels.numpy()) > 0).sum(axis=(1, 2, 3))
iou = (intersection + 1e-6) / (union + 1e-6)
print(f'📏 IoU médio no lote: {iou.mean():.4f}')

plt.figure(figsize=(18, 14))
print('\nLEGENDA: Esquerda=Satélite | Meio=Gabarito | Direita=Predição binária')

n_show = min(5, imgs.shape[0])
for i in range(n_show):
    plt.subplot(n_show, 3, i * 3 + 1)
    plt.imshow(imgs[i][:, :, 2], cmap='RdYlGn', vmin=0, vmax=1)
    plt.axis('off')
    if i == 0:
        plt.title('Satélite (NDVI)')

    plt.subplot(n_show, 3, i * 3 + 2)
    plt.imshow(labels[i][:, :, 0], cmap='binary_r')
    plt.axis('off')
    if i == 0:
        plt.title('Gabarito Real')

    plt.subplot(n_show, 3, i * 3 + 3)
    plt.imshow(preds_bin[i][:, :, 0], cmap='viridis')
    plt.axis('off')
    if i == 0:
        plt.title('Predição (>0.5)')

saida_fig = os.path.join(pasta_base, 'prova_real_validacao.png')
plt.tight_layout()
plt.savefig(saida_fig, dpi=200, bbox_inches='tight')
plt.show()
print(f'🖼️ Prova real salva em: {saida_fig}')


In [ ]:
# 5) Treinar
!python treinamento_seguro_unet.py

In [ ]:
# 6) Prova real (gera e salva a figura)
!python validacao_visual_modelo.py

In [ ]:
# 7) Exibir a prova real salva
from IPython.display import Image, display
import os
img_path = '/content/drive/MyDrive/Tese_IA_Jussara/prova_real_validacao.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
    print(f'✅ Exibindo: {img_path}')
else:
    print(f'⚠️ Imagem não encontrada em: {img_path}')

In [ ]:
# 8) Gravar script de teste de limiares
%%writefile teste_limiares_talhoes.py
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import glob
import numpy as np
import tensorflow as tf


def garantir_drive_montado() -> str:
    pasta_base = '/content/drive/MyDrive/Tese_IA_Jussara'
    if os.path.exists('/content/drive'):
        return pasta_base
    raise RuntimeError(
        "Drive não montado. No Colab, execute antes:\n"
        "from google.colab import drive\n"
        "drive.mount('/content/drive')"
    )


pasta_base = garantir_drive_montado()
modelo_path = os.path.join(pasta_base, 'Modelo_UNet_Jussara_2025_FINAL_v2.keras')
if not os.path.exists(modelo_path):
    raise FileNotFoundError(f'Modelo não encontrado: {modelo_path}')

busca = glob.glob(os.path.join(pasta_base, '*MASSIVE*tfrecord*'))
if not busca:
    busca = glob.glob(os.path.join(pasta_base, '*tfrecord*'))
if not busca:
    raise FileNotFoundError(f'Nenhum TFRecord encontrado em: {pasta_base}')

caminho_dados = sorted(busca, key=os.path.getmtime, reverse=True)[0]
print(f'📂 Dados usados no teste: {caminho_dados}')

KERNEL_SIZE = 128
READ_SIZE = 129
INPUT_BANDS = ['R_1', 'NIR_1', 'NDVI_1', 'R_2', 'NIR_2', 'NDVI_2']
LABEL_BAND = 'label_chip'
BATCH_SIZE = 16
N_BATCHES = 20
THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]


def parse_fast(example_proto):
    features = {band: tf.io.VarLenFeature(tf.float32) for band in INPUT_BANDS + [LABEL_BAND]}
    parsed = tf.io.parse_single_example(example_proto, features)

    xs = []
    for band in INPUT_BANDS:
        dense = tf.sparse.to_dense(parsed[band], default_value=0.0)
        img = tf.reshape(dense, [READ_SIZE, READ_SIZE, 1])
        img = tf.image.resize_with_crop_or_pad(img, KERNEL_SIZE, KERNEL_SIZE)
        min_v = tf.reduce_min(img)
        max_v = tf.reduce_max(img)
        img = tf.where(max_v > min_v, (img - min_v) / (max_v - min_v + 1e-6), tf.zeros_like(img))
        xs.append(img)

    image = tf.concat(xs, axis=-1)
    lbl = tf.sparse.to_dense(parsed[LABEL_BAND], default_value=0.0)
    lbl = tf.reshape(lbl, [READ_SIZE, READ_SIZE, 1])
    lbl = tf.image.resize_with_crop_or_pad(lbl, KERNEL_SIZE, KERNEL_SIZE)
    lbl = tf.cast(lbl > 0.5, tf.float32)
    return image, lbl


def metricas_binarias(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(np.uint8)
    y_true = y_true.astype(np.uint8)

    tp = np.logical_and(y_pred == 1, y_true == 1).sum()
    fp = np.logical_and(y_pred == 1, y_true == 0).sum()
    fn = np.logical_and(y_pred == 0, y_true == 1).sum()

    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    f1 = 2 * precision * recall / (precision + recall + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    return precision, recall, f1, iou


print('🤖 Carregando modelo...')
model = tf.keras.models.load_model(modelo_path, compile=False)

print('📦 Preparando lote de validação para sweep de limiar...')
dataset = tf.data.TFRecordDataset(caminho_dados, compression_type='GZIP').map(parse_fast).batch(BATCH_SIZE).take(N_BATCHES)

all_probs, all_labels = [], []
for imgs, labels in dataset:
    probs = model.predict(imgs, verbose=0)
    all_probs.append(probs)
    all_labels.append(labels.numpy())

if not all_probs:
    raise RuntimeError('Nenhum batch lido para teste de limiares.')

y_prob = np.concatenate(all_probs, axis=0)
y_true = np.concatenate(all_labels, axis=0)

print(f'✅ Amostras avaliadas: {y_prob.shape[0]}')
print('\n📊 Resultado por limiar:')
print('thr\tprecision\trecall\tf1\tiou')

best = None
for thr in THRESHOLDS:
    p, r, f1, iou = metricas_binarias(y_true, y_prob, thr)
    print(f'{thr:.2f}\t{p:.4f}\t\t{r:.4f}\t{f1:.4f}\t{iou:.4f}')
    if best is None or f1 > best['f1']:
        best = {'thr': thr, 'precision': p, 'recall': r, 'f1': f1, 'iou': iou}

print('\n🏆 Melhor limiar (por F1):')
print(best)

out_path = os.path.join(pasta_base, 'resultado_teste_limiares.txt')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write('thr\tprecision\trecall\tf1\tiou\n')
    for thr in THRESHOLDS:
        p, r, f1, iou = metricas_binarias(y_true, y_prob, thr)
        f.write(f'{thr:.2f}\t{p:.6f}\t{r:.6f}\t{f1:.6f}\t{iou:.6f}\n')
    f.write(f"\nmelhor_thr={best['thr']:.2f}\n")

print(f'📝 Resultado salvo em: {out_path}')


In [ ]:
# 9) Testar limiares (qual threshold é melhor para talhões?)
!python teste_limiares_talhoes.py